# 02 — Model Training
**Goal:** Split windows by subject (shuffled, seed 42), normalize with train stats,
train 3 Keras models (1D-CNN, GRU, CNN-LSTM) for both accel-only and accel+gyro,
and report a full comparison table.

Models are evaluated on the test set **only once** (after selecting the best model on validation).

In [1]:
# ── Standard-library imports ──────────────────────────────────────────────────
import os          # file and path operations
import json        # saving normalization statistics
import random      # Python built-in RNG (for seeding)
import warnings    # suppress noisy warnings
warnings.filterwarnings('ignore')

# ── Numeric / ML imports ──────────────────────────────────────────────────────
import numpy as np                        # array operations
import pandas as pd                       # results table
import matplotlib.pyplot as plt           # plotting
import matplotlib                         # for non-interactive backend
matplotlib.use('Agg')                     # no GUI window needed when running via nbconvert
import seaborn as sns                     # heatmap for confusion matrices

from sklearn.metrics import (
    accuracy_score,           # overall accuracy
    f1_score,                 # macro F1 and per-class F1
    confusion_matrix          # confusion matrix arrays
)

# ── TensorFlow / Keras imports ────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print('TensorFlow version:', tf.__version__)
print('Keras version     :', keras.__version__)

TensorFlow version: 2.19.0
Keras version     : 3.9.2


In [2]:
# ── Fix ALL random seeds for reproducibility ──────────────────────────────────
SEED = 42
# Python built-in seed
random.seed(SEED)
# NumPy seed
np.random.seed(SEED)
# TensorFlow / Keras seed (covers weight initializers and dropout)
tf.random.set_seed(SEED)
# Environment variable that makes TF operations fully deterministic
os.environ['PYTHONHASHSEED'] = str(SEED)
print(f'All random seeds fixed to {SEED}')

All random seeds fixed to 42


In [3]:
# ── Path constants ────────────────────────────────────────────────────────────
DATA_PATH    = os.path.join('..', 'data', 'windows.npz')   # preprocessed data
MODELS_DIR   = os.path.join('..', 'models')                # save norm stats + best model
FIGURES_DIR  = os.path.join('..', 'figures')               # save confusion matrices + csv

# Create output directories if they do not exist
os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# Activity label names for plot axes
CLASS_NAMES = ['Walking', 'Jogging', 'Stairs', 'Sitting', 'Standing']

In [4]:
# ── Load the preprocessed windows ─────────────────────────────────────────────
data     = np.load(DATA_PATH)
X        = data['X']         # shape (N, 100, 6) — ax ay az gx gy gz
y        = data['y']         # shape (N,) — integer labels 0-4
subjects = data['subjects']  # shape (N,) — subject IDs per window

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'subjects shape: {subjects.shape}')
print(f'Unique subjects: {np.unique(subjects).tolist()}')

X shape: (17685, 100, 6)
y shape: (17685,)
subjects shape: (17685,)


Unique subjects: [1600, 1601, 1602, 1603, 1604, 1605, 1606, 1607, 1608, 1609, 1610, 1611, 1612, 1613, 1614, 1615, 1616, 1617, 1618, 1619, 1620, 1621, 1622, 1623, 1624, 1625, 1626, 1627, 1628, 1629, 1630, 1631, 1632, 1633, 1634, 1635, 1636, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1644, 1645, 1646, 1647, 1648, 1649, 1650]


In [5]:
# ── Subject-wise train / validation / test split ──────────────────────────────
# We shuffle the 51 unique subject IDs with seed 42 to avoid ordering bias,
# then assign the first ~70% to train, next ~15% to val, last ~15% to test.
# This guarantees NO subject appears in more than one split.

unique_subjects = np.unique(subjects)    # sorted array of 51 IDs
n_subjects      = len(unique_subjects)   # 51

# Shuffle subject IDs with the fixed seed
rng = np.random.default_rng(SEED)
shuffled_subjects = rng.permutation(unique_subjects)

# Compute split indices
n_train = int(np.floor(0.70 * n_subjects))   # 70% → 35 subjects
n_val   = int(np.floor(0.15 * n_subjects))   # 15% →  7 subjects
# remainder goes to test (51 - 35 - 7 = 9 subjects → ~18%)

# Slice the shuffled list
train_subjects = shuffled_subjects[:n_train]
val_subjects   = shuffled_subjects[n_train : n_train + n_val]
test_subjects  = shuffled_subjects[n_train + n_val :]

# Print every subject ID in each split to prove no overlap
print(f'Train subjects ({len(train_subjects)}): {sorted(train_subjects.tolist())}')
print(f'Val   subjects ({len(val_subjects)}): {sorted(val_subjects.tolist())}')
print(f'Test  subjects ({len(test_subjects)}): {sorted(test_subjects.tolist())}')

# Sanity check: intersection of any two sets must be empty
assert len(set(train_subjects) & set(val_subjects))  == 0, 'Leak: train ∩ val!'
assert len(set(train_subjects) & set(test_subjects)) == 0, 'Leak: train ∩ test!'
assert len(set(val_subjects)   & set(test_subjects)) == 0, 'Leak: val ∩ test!'
print('\n✅ No subject overlap between splits.')

Train subjects (35): [1603, 1604, 1605, 1607, 1609, 1610, 1615, 1616, 1617, 1618, 1620, 1621, 1623, 1624, 1625, 1626, 1627, 1628, 1629, 1630, 1631, 1632, 1634, 1637, 1638, 1639, 1640, 1641, 1642, 1643, 1645, 1646, 1647, 1648, 1650]
Val   subjects (7): [1600, 1606, 1611, 1619, 1622, 1635, 1644]
Test  subjects (9): [1601, 1602, 1608, 1612, 1613, 1614, 1633, 1636, 1649]

✅ No subject overlap between splits.


In [6]:
# ── Create boolean masks and index arrays for each split ──────────────────────
# A window belongs to a split if its subject ID is in the corresponding set
train_mask = np.isin(subjects, train_subjects)  # True where subject is in train
val_mask   = np.isin(subjects, val_subjects)    # True where subject is in val
test_mask  = np.isin(subjects, test_subjects)   # True where subject is in test

# Apply masks to get arrays for each split
X_train, y_train = X[train_mask], y[train_mask]
X_val,   y_val   = X[val_mask],   y[val_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f'Train windows: {len(X_train):,}')
print(f'Val   windows: {len(X_val):,}')
print(f'Test  windows: {len(X_test):,}')

Train windows: 12,065
Val   windows: 2,455
Test  windows: 3,165


In [7]:
# ── Compute normalisation statistics from TRAIN set only ──────────────────────
# We compute mean and standard deviation over the training windows
# across both the time dimension (100 steps) and the window dimension (N_train),
# resulting in one mean and one std per channel (6 values each).
# Using only train stats prevents any test-set leakage.

# Reshape to (N_train * 100, 6) then compute per-channel statistics
X_train_flat = X_train.reshape(-1, X_train.shape[-1])   # shape: (N_train*100, 6)
train_mean   = X_train_flat.mean(axis=0)                 # shape: (6,)
train_std    = X_train_flat.std(axis=0)                  # shape: (6,)

# Guard against a zero std (would cause division by zero)
train_std[train_std == 0] = 1.0

# Channel names for the JSON file
CHANNEL_NAMES = ['ax', 'ay', 'az', 'gx', 'gy', 'gz']

# Save statistics to JSON so they can be loaded at inference time
norm_stats = {
    'channels' : CHANNEL_NAMES,
    'mean'     : train_mean.tolist(),
    'std'      : train_std.tolist()
}
norm_path = os.path.join(MODELS_DIR, 'norm_stats.json')
with open(norm_path, 'w') as f:
    json.dump(norm_stats, f, indent=2)

print('Normalisation stats (train set only):')
for i, ch in enumerate(CHANNEL_NAMES):
    print(f'  {ch}: mean={train_mean[i]:8.4f}, std={train_std[i]:8.4f}')
print(f'\nSaved to: {os.path.abspath(norm_path)}')

Normalisation stats (train set only):
  ax: mean=  1.0253, std=  6.4385
  ay: mean= -1.7554, std=  8.1821
  az: mean= -0.2430, std=  4.6221
  gx: mean= -0.0112, std=  1.2862
  gy: mean= -0.0079, std=  1.5038
  gz: mean= -0.0020, std=  1.0705

Saved to: C:\Users\shiva\Downloads\HAR2\models\norm_stats.json


In [8]:
# ── Apply normalisation to all splits ─────────────────────────────────────────
# Subtract the train mean and divide by train std for every split.
# Broadcasting: mean/std have shape (6,) which broadcasts over (N, 100, 6).

X_train_n = (X_train - train_mean) / train_std   # normalised train
X_val_n   = (X_val   - train_mean) / train_std   # normalised val (train stats)
X_test_n  = (X_test  - train_mean) / train_std   # normalised test (train stats)

print(f'Normalised train range: [{X_train_n.min():.2f}, {X_train_n.max():.2f}]')
print(f'Normalised val   range: [{X_val_n.min():.2f},   {X_val_n.max():.2f}]')
print(f'Normalised test  range: [{X_test_n.min():.2f},  {X_test_n.max():.2f}]')

Normalised train range: [-32.68, 21.03]


Normalised val   range: [-13.19,   17.66]
Normalised test  range: [-11.26,  8.35]


In [9]:
# ── Helper: select channel subset ─────────────────────────────────────────────
# Channels 0-2 are accel (ax, ay, az); channels 3-5 are gyro (gx, gy, gz).
# For the accel-only experiment we take only the first 3 channels.

def get_channels(X_tr, X_v, X_te, mode):
    """Return (X_train, X_val, X_test, n_channels) for the chosen sensor mode."""
    # 'accel' mode: slice only the first three channels
    if mode == 'accel':
        return X_tr[:, :, :3], X_v[:, :, :3], X_te[:, :, :3], 3
    # 'both' mode: use all six channels
    elif mode == 'both':
        return X_tr, X_v, X_te, 6
    else:
        raise ValueError(f'Unknown mode: {mode}')

In [10]:
# ── Model 1: 1D-CNN ───────────────────────────────────────────────────────────
# Two Conv1D layers extract local temporal patterns, followed by GlobalAveragePooling
# which compresses the time dimension, then a Dense classifier.
# Kept small (< 100k parameters) by using narrow filter counts.

def build_cnn(input_shape, n_classes=5):
    """Build and compile a small 1D-CNN.

    Architecture:
      Conv1D(32, 5) → BatchNorm → ReLU
      Conv1D(64, 3) → BatchNorm → ReLU
      GlobalAveragePooling1D
      Dense(64, ReLU) → Dropout(0.3)
      Dense(n_classes, softmax)
    """
    # Use Keras functional API for clarity
    inp = keras.Input(shape=input_shape, name='input')      # (100, n_channels)

    # First convolutional block: 32 filters, kernel size 5, ReLU activation
    x = layers.Conv1D(32, kernel_size=5, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)   # normalise activations for stable training
    x = layers.Activation('relu')(x)

    # Second convolutional block: 64 filters, kernel size 3, ReLU activation
    x = layers.Conv1D(64, kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Global average pooling collapses the 100 time steps to a single vector
    x = layers.GlobalAveragePooling1D()(x)

    # Fully connected layer with dropout to reduce overfitting
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Output layer: one neuron per activity class with softmax probabilities
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    # Compile with Adam optimiser and sparse categorical cross-entropy loss
    model = keras.Model(inputs=inp, outputs=out, name='CNN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick check: build with 6 channels to count parameters
cnn_test = build_cnn((100, 6))
print(f'CNN parameter count: {cnn_test.count_params():,}')

CNN parameter count: 11,973


In [11]:
# ── Model 2: GRU ──────────────────────────────────────────────────────────────
# A two-layer Gated Recurrent Unit network. GRUs are good at capturing
# long-range temporal dependencies in the 5-second window.
# Stacked: GRU(64) → GRU(32) keeps parameter count well under 100k.

def build_gru(input_shape, n_classes=5):
    """Build and compile a stacked GRU model.

    Architecture:
      GRU(64, return_sequences=True)
      GRU(32)
      Dense(32, ReLU) → Dropout(0.3)
      Dense(n_classes, softmax)
    """
    inp = keras.Input(shape=input_shape, name='input')   # (100, n_channels)

    # First GRU layer: returns sequences so the second GRU sees all time steps
    x = layers.GRU(64, return_sequences=True)(inp)

    # Second GRU layer: returns only the final hidden state (shape: 32,)
    x = layers.GRU(32)(x)

    # Dense head with dropout
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Softmax output over 5 classes
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    # Compile the model
    model = keras.Model(inputs=inp, outputs=out, name='GRU')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick check: parameter count
gru_test = build_gru((100, 6))
print(f'GRU parameter count: {gru_test.count_params():,}')

GRU parameter count: 24,453


In [12]:
# ── Model 3: CNN-LSTM ─────────────────────────────────────────────────────────
# A hybrid: a Conv1D block extracts local features, then an LSTM captures
# sequential patterns across the extracted features.
# This architecture is popular in HAR literature.

def build_cnn_lstm(input_shape, n_classes=5):
    """Build and compile a CNN-LSTM hybrid.

    Architecture:
      Conv1D(32, 5) → BatchNorm → ReLU
      Conv1D(32, 3) → BatchNorm → ReLU
      LSTM(64)
      Dense(32, ReLU) → Dropout(0.3)
      Dense(n_classes, softmax)
    """
    inp = keras.Input(shape=input_shape, name='input')   # (100, n_channels)

    # First Conv1D block: captures low-level local patterns
    x = layers.Conv1D(32, kernel_size=5, padding='same', use_bias=False)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Second Conv1D block: captures higher-level local patterns
    x = layers.Conv1D(32, kernel_size=3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # LSTM reads the sequence of CNN-extracted features over 100 time steps
    x = layers.LSTM(64)(x)

    # Dense head with dropout
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)

    # Softmax output
    out = layers.Dense(n_classes, activation='softmax', name='output')(x)

    # Compile
    model = keras.Model(inputs=inp, outputs=out, name='CNN_LSTM')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# Quick check: parameter count
cnn_lstm_test = build_cnn_lstm((100, 6))
print(f'CNN-LSTM parameter count: {cnn_lstm_test.count_params():,}')

CNN-LSTM parameter count: 31,365


In [13]:
# ── Print a summary of all three model architectures ──────────────────────────
# This summarises the layer stack, output shapes, and parameter count
for model_obj in [cnn_test, gru_test, cnn_lstm_test]:
    print('=' * 60)
    model_obj.summary()
    print()

Model: "CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 100, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 100, 32)        │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 100, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 100, 64)        │         6,144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 100, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,973 (46.77 KB)

 Trainable params: 11,781 (46.02 KB)

 Non-trainable params: 192 (768.00 B)

Model: "GRU"

... (5 outputs left out)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 100, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 100, 32)        │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 100, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 100, 32)        │         3,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 100, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 100, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 31,365 (122.52 KB)

 Trainable params: 31,237 (122.02 KB)

 Non-trainable params: 128 (512.00 B)

In [14]:
# ── Training helper function ──────────────────────────────────────────────────
# Encapsulates one full training run for any model + sensor mode combination.

EPOCHS     = 60     # maximum epochs; early stopping will typically stop sooner
BATCH_SIZE = 128    # mini-batch size; large enough for stable gradients

def train_model(build_fn, mode, label):
    """Train one model for one sensor mode and return metrics + trained model.

    Parameters
    ----------
    build_fn : callable that returns a compiled Keras model given input_shape
    mode     : 'accel' (3 channels) or 'both' (6 channels)
    label    : human-readable name for this run, used in printed output

    Returns
    -------
    model     : the trained Keras model (best weights restored by EarlyStopping)
    history   : training History object
    metrics   : dict with val_acc, test_acc, macro_f1, per_class_f1, params, size_kb
    """
    print(f'\n{"-"*60}')
    print(f'Training: {label}')
    print(f'{"-"*60}')

    # Reset the TF random seed before every run for reproducibility
    tf.random.set_seed(SEED)

    # Select the correct channel subset
    Xtr, Xv, Xte, n_ch = get_channels(X_train_n, X_val_n, X_test_n, mode)

    # Build a fresh model with the correct input shape
    input_shape = (Xtr.shape[1], n_ch)   # (100, n_channels)
    model = build_fn(input_shape)

    # Print parameter count for this run
    n_params = model.count_params()
    print(f'Parameters: {n_params:,}')

    # EarlyStopping: monitor val_loss; stop if no improvement for 10 epochs;
    # restore the weights from the epoch with the best val_loss.
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,   # ensures we evaluate the best checkpoint
        verbose=1
    )

    # Train the model
    history = model.fit(
        Xtr, y_train,
        validation_data=(Xv, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=2   # one line per epoch
    )

    # ── Evaluate on VALIDATION set (for model selection) ──────────────────
    # We choose the best model by validation performance, NOT test performance.
    val_preds = model.predict(Xv, verbose=0).argmax(axis=1)
    val_acc   = accuracy_score(y_val, val_preds)         # validation accuracy
    val_f1    = f1_score(y_val, val_preds, average='macro')  # validation macro F1
    print(f'Val accuracy: {val_acc:.4f}  |  Val macro F1: {val_f1:.4f}')

    # ── Evaluate on TEST set (used ONCE here, not for selection) ──────────
    test_preds    = model.predict(Xte, verbose=0).argmax(axis=1)
    test_acc      = accuracy_score(y_test, test_preds)               # test accuracy
    macro_f1      = f1_score(y_test, test_preds, average='macro')    # macro F1
    per_class_f1  = f1_score(y_test, test_preds, average=None)       # per-class F1 array
    print(f'Test accuracy: {test_acc:.4f}  |  Test macro F1: {macro_f1:.4f}')

    # ── Model size in KB ──────────────────────────────────────────────────
    # Save to a temp file, measure its size, then remove the temp file
    tmp_path = os.path.join(MODELS_DIR, '_tmp_model.keras')
    model.save(tmp_path)
    size_kb = os.path.getsize(tmp_path) / 1024
    os.remove(tmp_path)   # clean up temporary file
    print(f'Model size: {size_kb:.1f} KB')

    # ── Save confusion matrix image ────────────────────────────────────────
    cm = confusion_matrix(y_test, test_preds)   # shape (5, 5)
    fig, ax = plt.subplots(figsize=(6, 5))
    # Normalise by row so the heatmap shows per-class recall
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=True, fmt='.2f', cmap='Blues',
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax
    )
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_title(f'{label}\nTest accuracy: {test_acc:.3f}  Macro F1: {macro_f1:.3f}')
    plt.tight_layout()

    # Build a safe filename from the run label
    fig_name = label.replace(' ', '_').replace('+', 'plus') + '.png'
    fig_path = os.path.join(FIGURES_DIR, fig_name)
    fig.savefig(fig_path, dpi=120)
    plt.close(fig)
    print(f'Confusion matrix saved: {fig_path}')

    # Collect all metrics in a dictionary for the results table
    metrics = {
        'val_acc'      : val_acc,
        'val_f1'       : val_f1,
        'test_acc'     : test_acc,
        'macro_f1'     : macro_f1,
        'per_class_f1' : per_class_f1.tolist(),
        'params'       : n_params,
        'size_kb'      : size_kb,
        'stopped_epoch': early_stop.stopped_epoch,
    }
    return model, history, metrics

In [15]:
# ── Run all 6 training experiments ────────────────────────────────────────────
# 3 models × 2 sensor modes = 6 runs
# We record every result; model selection for best_model.keras is on VAL accuracy.

EXPERIMENTS = [
    ('CNN',      build_cnn,      'accel'),   # run 1
    ('CNN',      build_cnn,      'both'),    # run 2
    ('GRU',      build_gru,      'accel'),   # run 3
    ('GRU',      build_gru,      'both'),    # run 4
    ('CNN_LSTM', build_cnn_lstm, 'accel'),   # run 5
    ('CNN_LSTM', build_cnn_lstm, 'both'),    # run 6
]

all_results = []    # list of (label, model, metrics) tuples

for arch_name, build_fn, sensor_mode in EXPERIMENTS:
    # Construct a human-readable label for this run
    sensor_label = 'Accel-only' if sensor_mode == 'accel' else 'Accel+Gyro'
    run_label    = f'{arch_name} ({sensor_label})'

    # Train and evaluate the model; get back the trained model and metrics dict
    model, history, metrics = train_model(build_fn, sensor_mode, run_label)

    # Store everything for the summary table
    all_results.append((run_label, model, metrics))


------------------------------------------------------------
Training: CNN (Accel-only)
------------------------------------------------------------
Parameters: 11,493


Epoch 1/60


95/95 - 3s - 34ms/step - accuracy: 0.6357 - loss: 0.9501 - val_accuracy: 0.6277 - val_loss: 1.0423


Epoch 2/60


95/95 - 1s - 14ms/step - accuracy: 0.7835 - loss: 0.5269 - val_accuracy: 0.5910 - val_loss: 0.8711


Epoch 3/60


... (221 outputs left out)

Epoch 18: early stopping


Restoring model weights from the end of the best epoch: 8.


Val accuracy: 0.8065  |  Val macro F1: 0.8042


Test accuracy: 0.8073  |  Test macro F1: 0.8090
Model size: 424.7 KB
Confusion matrix saved: ..\figures\CNN_LSTM_(AccelplusGyro).png


In [16]:
# ── Select the best model by VALIDATION accuracy ───────────────────────────────
# Iterate over all results and find the one with the highest val_acc.
# The test set has NOT been used for selection here.

best_label  = None
best_model  = None
best_val_acc = -1.0

for run_label, model, metrics in all_results:
    # Compare this run's validation accuracy to the current best
    if metrics['val_acc'] > best_val_acc:
        best_val_acc = metrics['val_acc']
        best_label   = run_label
        best_model   = model

print(f'Best model by validation accuracy: {best_label}')
print(f'  Val accuracy : {best_val_acc:.4f}')

# Save the best model to disk in Keras format
best_model_path = os.path.join(MODELS_DIR, 'best_model.keras')
best_model.save(best_model_path)
print(f'Saved to: {os.path.abspath(best_model_path)}')

Best model by validation accuracy: CNN (Accel-only)
  Val accuracy : 0.8204
Saved to: C:\Users\shiva\Downloads\HAR2\models\best_model.keras


In [17]:
# ── Build and print the full results table ────────────────────────────────────
# One row per run, with all requested metrics.

rows = []
for run_label, model, m in all_results:
    # Unpack per-class F1 into individual columns for readability
    pcf1 = m['per_class_f1']   # list of 5 floats
    row = {
        'Run'                : run_label,
        'Val Acc'            : round(m['val_acc'],  4),
        'Val Macro F1'       : round(m['val_f1'],   4),
        'Test Acc'           : round(m['test_acc'], 4),
        'Test Macro F1'      : round(m['macro_f1'], 4),
        'F1 Walking'         : round(pcf1[0], 4),
        'F1 Jogging'         : round(pcf1[1], 4),
        'F1 Stairs'          : round(pcf1[2], 4),
        'F1 Sitting'         : round(pcf1[3], 4),
        'F1 Standing'        : round(pcf1[4], 4),
        'Params'             : m['params'],
        'Size (KB)'          : round(m['size_kb'], 1),
        'Stopped Epoch'      : m['stopped_epoch'],
    }
    rows.append(row)

# Build a DataFrame from the rows list
results_df = pd.DataFrame(rows)

# Display the full table in the notebook
pd.set_option('display.max_columns', None)   # show every column
pd.set_option('display.width', 200)          # wider display
print('\n===== FULL RESULTS TABLE =====')
print(results_df.to_string(index=False))


===== FULL RESULTS TABLE =====
                  Run  Val Acc  Val Macro F1  Test Acc  Test Macro F1  F1 Walking  F1 Jogging  F1 Stairs  F1 Sitting  F1 Standing  Params  Size (KB)  Stopped Epoch
     CNN (Accel-only)   0.8204        0.8230    0.7930         0.7956      0.6185      0.9192     0.5588      0.9383       0.9430   11493      185.5             17
     CNN (Accel+Gyro)   0.8179        0.8171    0.8367         0.8393      0.7609      0.8252     0.7275      0.9383       0.9445   11973      191.1             22
     GRU (Accel-only)   0.7646        0.7637    0.7599         0.7667      0.5159      0.8212     0.6120      0.9390       0.9453   23877      321.3             14
     GRU (Accel+Gyro)   0.7161        0.7124    0.8054         0.8049      0.6924      0.8458     0.6044      0.9391       0.9429   24453      328.0             12
CNN_LSTM (Accel-only)   0.7760        0.7770    0.8588         0.8608      0.7833      0.8864     0.7538      0.9376       0.9429   30885      419.1

In [18]:
# ── Save results table to CSV ─────────────────────────────────────────────────
csv_path = os.path.join(FIGURES_DIR, 'results.csv')
results_df.to_csv(csv_path, index=False)
print(f'Results table saved to: {os.path.abspath(csv_path)}')

Results table saved to: C:\Users\shiva\Downloads\HAR2\figures\results.csv


In [19]:
# ── Final summary ─────────────────────────────────────────────────────────────
# Highlight the best-performing model and confirm test set was used only once.

best_row = results_df.loc[results_df['Val Acc'].idxmax()]
print('\n===== BEST MODEL SUMMARY =====')
print(f'  Model          : {best_row["Run"]}')
print(f'  Val Accuracy   : {best_row["Val Acc"]}')
print(f'  Val Macro F1   : {best_row["Val Macro F1"]}')
print(f'  Test Accuracy  : {best_row["Test Acc"]}')
print(f'  Test Macro F1  : {best_row["Test Macro F1"]}')
print(f'  Parameters     : {int(best_row["Params"]):,}')
print(f'  Model Size     : {best_row["Size (KB)"]} KB')
print(f'  Saved to       : {os.path.abspath(best_model_path)}')
print('\nNote: test metrics reported above are the ONLY time the test set was used.')


===== BEST MODEL SUMMARY =====
  Model          : CNN (Accel-only)
  Val Accuracy   : 0.8204
  Val Macro F1   : 0.823
  Test Accuracy  : 0.793
  Test Macro F1  : 0.7956
  Parameters     : 11,493
  Model Size     : 185.5 KB
  Saved to       : C:\Users\shiva\Downloads\HAR2\models\best_model.keras

Note: test metrics reported above are the ONLY time the test set was used.
